# 데이터 분석 v2 — IV / 그룹기여도(leave-one-out) / EXT상관 (83개 피처)

원본(팀원 코드, model_dataset.csv 47개 변수 기준)을 model_dataset_v2.csv(85컬럼, 83개 피처) 기준으로 수정.

**변경 사항**
- step1: A_금융에 신규 13개(신용조회횟수6+1, bureau집계6) 추가, **C_비금융_신규 그룹(17개) 신설**
  (기존엔 A/B/EXT 3분류였으나 RQ2 검증을 위해 B(기존 비금융)와 C(신규 비금융)를 구분)
- step2: 개념그룹 14개 → **22개**로 확장 (C군 6개 + A신규 2개 그룹 추가), map_5c에 신규 변수 매핑 추가
  - 사회연결망/통신관련은 5C 밖 신규축("기타")으로 명시 — 억지로 5C에 끼워맞추지 않음
  - bureau/신용조회횟수는 Character(상환의지) 축으로 매핑 (팀이 확인한 "Character 축 보강" 목적과 일치)

**실제 데이터로 스모크 테스트 완료** (축소판: 표본 5천/트리 20개, 에러 없이 정상 동작 확인)
⚠️ 아래는 실제 설정(표본 15만, 트리 250개, 22개 그룹)이라 **원본보다 실행시간이 더 걸립니다**
(그룹 수가 14→22로 늘어난 만큼 leave-one-out 반복 횟수도 늘어남. 콜랩에서 여유 있게 실행 권장)


**이번 수정**: 코랩에서 구글드라이브 파일을 못 찾던 경로 문제 수정 (상대경로 → `BASE_DIR` 절대경로). 그 외 로직은 전혀 건드리지 않음.

## 0. 구글드라이브 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1. 단변량 분석 — 설정 (경로/상수)

In [3]:
"""
step1_univariate.py — 사전 변수분석 (단변량) [v2: 85컬럼(83피처) 버전]

목적: M1~M5 모델링 전에, 각 변수가 TARGET과 얼마나 관련 있는지(IV),
      그 정보가 외부신용점수(EXT_SOURCE)와 겹치는지(상관)를 확인한다.

주 지표는 IV(Information Value).

변경사항 (v1 -> v2):
  - A_금융에 신규 13개 추가: AMT_REQ_CREDIT_BUREAU_*(6+결측플래그1), BUREAU_*(6, bureau.csv 집계)
  - C_비금융_신규 그룹 신설(17개): 기존엔 A/B/EXT 3분류였으나, C군(현재 금융권 미사용 비금융)을
    B군(기존 비금융)과 구분해야 RQ2 검증이 가능하므로 분리

입력 : model_dataset_v2.csv (85컬럼, C군+bureau 병합 완료본)
출력 : outputs_uni/univariate_summary.csv  (변수별 IV·검정·EXT상관 통합표)
"""

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content/drive/MyDrive/BOOSTMAP/데이터/원본")  # 본인 환경에 맞게 수정

DATA = BASE_DIR / "model_dataset_v2.csv"
OUT = BASE_DIR / "outputs_uni"; OUT.mkdir(exist_ok=True)

TARGET = "TARGET"
ID = "SK_ID_CURR"
EXT_COLS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
N_BINS = 10        # 연속변수 구간 수
EPS = 0.5          # WOE 라플라스 보정(0 셀 방지)

### 변수 그룹 정의 (A_금융 / C_비금융_신규)

In [4]:
# 금융(A군) 변수 명시 목록 — 기존 14개
A_FEATURES_ORIG = {"AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
                    "NAME_CONTRACT_TYPE_BIN", "DTI", "LTV_GOODS", "CREDIT_TO_INCOME",
                    "ANNUITY_TO_CREDIT", "AMT_CREDIT_LOG", "AMT_ANNUITY_LOG",
                    "AMT_GOODS_PRICE_LOG", "AMT_INCOME_TOTAL_LOG", "AMT_INCOME_OUTLIER"}

# 금융(A군) 신규 13개 — 신용조회횟수(6+결측플래그1) + bureau.csv 집계(6)
# 신용조회횟수는 "C군으로 재검토 예정(향후 과제)"로 남겨두기로 했으나,
# 현재는 팀 결정대로 A_금융 유지
A_FEATURES_NEW = {"AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY", "AMT_REQ_CREDIT_BUREAU_WEEK",
                  "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR",
                  "AMT_REQ_CREDIT_BUREAU_MISSING_FLAG",
                  "BUREAU_LOAN_COUNT", "BUREAU_ACTIVE_LOAN_COUNT", "BUREAU_DAYS_OVERDUE_MAX",
                  "BUREAU_DAYS_OVERDUE_MEAN", "BUREAU_DEBT_RATIO", "BUREAU_NO_HISTORY_FLAG"}

A_FEATURES = A_FEATURES_ORIG | A_FEATURES_NEW

# C군(신규 비금융, 17개) — 원본 6개 + 원핫 접두사 2개(교육수준5+주거유형6)
C_FEATURES_RAW = {"FLAG_OWN_CAR", "CNT_CHILDREN", "FLAG_CONT_MOBILE",
                  "OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE", "SOCIAL_CIRCLE_MISSING_FLAG"}
C_PREFIXES = ("NAME_EDUCATION_TYPE_C_", "NAME_HOUSING_TYPE_C_")

### 변수 분류 함수 (classify_columns)

In [5]:
# --------------------------------------------------------- 변수 분류
def classify_columns(df):
    ext = set(EXT_COLS) | {c + "_MISSING" for c in EXT_COLS}
    rows = []
    for c in df.columns:
        if c in (ID, TARGET):
            continue
        if c in ext:
            grp = "EXT"
        elif c in A_FEATURES:
            grp = "A_금융"
        elif c in C_FEATURES_RAW or c.startswith(C_PREFIXES):
            grp = "C_비금융_신규"
        else:
            grp = "B_비금융_기존"
        u = df[c].dropna().unique()
        is_bin = set(np.unique(u)).issubset({0, 1}) and len(u) <= 2
        rows.append({"feature": c, "group": grp,
                     "vtype": "binary" if is_bin else "continuous"})
    return pd.DataFrame(rows)

### WOE / IV 계산 함수

In [6]:
# --------------------------------------------------------- WOE / IV
def _woe_iv_from_groups(g, y):
    df = pd.DataFrame({"g": g, "y": y})
    tot_bad = df.y.sum(); tot_good = len(df) - tot_bad
    agg = df.groupby("g").agg(n=("y", "size"), bad=("y", "sum"))
    agg["good"] = agg.n - agg.bad
    agg["bad_dist"] = (agg.bad + EPS) / (tot_bad + EPS * len(agg))
    agg["good_dist"] = (agg.good + EPS) / (tot_good + EPS * len(agg))
    agg["woe"] = np.log(agg.good_dist / agg.bad_dist)
    agg["iv_part"] = (agg.good_dist - agg.bad_dist) * agg.woe
    agg["bad_rate"] = agg.bad / agg.n
    return float(agg.iv_part.sum()), agg.reset_index()


def woe_iv_binary(x, y):
    return _woe_iv_from_groups(x.fillna(-1), y)


def woe_iv_continuous(x, y, n_bins=N_BINS):
    # 결측은 반드시 별도 구간으로 유지 (결측 자체가 신호)
    x = x.replace([np.inf, -np.inf], np.nan)
    binned = pd.Series(index=x.index, dtype=object)
    notna = x.notna()
    try:
        binned[notna] = pd.qcut(x[notna], n_bins, duplicates="drop").astype(str)
    except ValueError:
        binned[notna] = pd.cut(x[notna], min(n_bins, x[notna].nunique())).astype(str)
    binned[~notna] = "MISSING"
    return _woe_iv_from_groups(binned, y)


def iv_strength(iv):
    if iv < 0.02: return "무용"
    if iv < 0.1:  return "약함"
    if iv < 0.3:  return "중간"
    if iv < 0.5:  return "강함"
    return "매우강함(과적합 의심)"

### 유의성 검정 (카이제곱 / Mann-Whitney)

In [8]:
# --------------------------------------------------------- 유의성 검정
def chi2_binary(x, y):
    ct = pd.crosstab(x.fillna(-1), y)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))     # Cramér's V
    return chi2, p, v


def mwu_continuous(x, y):
    # 소득/대출액은 극단적 우편향 → 정규성 가정 없는 Mann-Whitney가 안전
    x = x.replace([np.inf, -np.inf], np.nan)
    a = x[(y == 0) & x.notna()]; b = x[(y == 1) & x.notna()]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    u, p_mwu = stats.mannwhitneyu(a, b, alternative="two-sided")
    rbc = 1 - 2 * u / (len(a) * len(b))               # rank-biserial 효과크기
    t, p_t = stats.ttest_ind(a, b, equal_var=False)   # 참고용 Welch t
    return p_mwu, rbc, p_t

### EXT_SOURCE 상관관계 (신규성 확인)

In [9]:
# --------------------------------------------------------- EXT 상관(신규성)
def ext_correlation(x, df, vtype):
    # EXT_SOURCE_2(결측 0.2%로 가장 온전)와의 |상관|. 낮을수록 새로운 정보.
    ext = df["EXT_SOURCE_2"]
    m = ext.notna() & x.notna() & np.isfinite(x.replace([np.inf, -np.inf], np.nan))
    if m.sum() < 100:
        return np.nan
    if vtype == "continuous":
        r, _ = stats.spearmanr(x[m], ext[m])
    else:
        r, _ = stats.pointbiserialr(x[m].astype(int), ext[m])
    return abs(r)

### 실행 (main)

In [10]:
# --------------------------------------------------------- 메인
def main():
    df = pd.read_csv(DATA)
    y = df[TARGET].values
    print(f"{df.shape[0]:,}행, 부도율 {y.mean():.2%}")

    cls = classify_columns(df)
    print(cls.group.value_counts().to_string())

    summary, woe_tables = [], {}
    for _, row in cls.iterrows():
        f, grp, vt = row.feature, row.group, row.vtype
        x = df[f]
        if vt == "binary":
            iv, wtab = woe_iv_binary(x, y)
            _, p_assoc, effect = chi2_binary(x, y); test = "chi2"
        else:
            iv, wtab = woe_iv_continuous(x, y)
            p_assoc, effect, _ = mwu_continuous(x, y); test = "mann_whitney"
        woe_tables[f] = wtab
        ext_corr = np.nan if grp == "EXT" else ext_correlation(x, df, vt)
        summary.append({"feature": f, "group": grp, "vtype": vt,
                        "IV": round(iv, 4), "IV_강도": iv_strength(iv),
                        "검정": test, "p_value": p_assoc,
                        "효과크기": round(effect, 4) if pd.notna(effect) else np.nan,
                        "EXT2_상관": round(ext_corr, 4) if pd.notna(ext_corr) else np.nan})

    S = pd.DataFrame(summary).sort_values("IV", ascending=False)
    S["유의_5pct"] = S.p_value < 0.05
    S.to_csv(OUT / "univariate_summary.csv", index=False, encoding="utf-8-sig")
    print(f"\n저장 완료: {OUT / 'univariate_summary.csv'} ({len(S)}개 변수)")


if __name__ == "__main__":
    main()

307,511행, 부도율 8.07%
group
B_비금융_기존    33
A_금융        27
C_비금융_신규    17
EXT          6

저장 완료: /content/drive/MyDrive/BOOSTMAP/데이터/원본/outputs_uni/univariate_summary.csv (83개 변수)


## Step 2. 그룹기여도(leave-one-out) & 최종 변수선정 — 설정

In [16]:
"""
step2_group_contribution.py — 그룹 단위 조건부 기여도 & 최종 변수선정 [v2: 85컬럼 버전]

배경: IV는 변수를 '혼자' 평가하지만, 실제 모델은 변수를 조합해서 쓴다.
      그래서 leave-one-out(하나 빼고 AUC 재측정)으로 '다른 변수가 다 있을 때의
      추가 기여'를 측정한다. 이게 중복 변수를 걸러낸다.

핵심 원칙:
  1. 원핫 더미(직업/업종/소득유형/교육수준/주거유형)는 개별이 아니라 '그룹 통째로' 넣고 뺀다.
  2. 원본 금액(AMT_*)은 비율변수(ANNUITY_TO_CREDIT 등)와 정보가 겹친다 → 대표 2개만 유지.
  3. (v2 신규) 상관관계 높은 쌍(EDA에서 발견, |r|>=0.9)의 그룹기여도를 특히 주의 깊게 볼 것:
     - DAYS_EMPLOYED_ANOM ↔ NAME_INCOME_TYPE_G_Pensioner (r=0.9996)
     - AMT_REQ_CREDIT_BUREAU_MISSING_FLAG ↔ BUREAU_NO_HISTORY_FLAG (r=0.967)
     서로 다른 개념그룹에 속해 있어서, 하나가 낮은 기여도로 나오면 중복 신호로 해석 가능.

변경사항 (v1 -> v2):
  - build_groups에 C군 개념그룹 6개 신규: 사회연결망, 통신, 자녀수, 차량보유, 교육수준, 주거유형
  - build_groups에 A_금융 신규 개념그룹 2개: 신용조회횟수(AMT_REQ), bureau이력
  - map_5c에 신규 변수 매핑 추가. bureau/AMT_REQ는 Character(상환의지) 축으로,
    사회연결망/통신은 5C 밖 신규축("기타")으로 명시 — 팀 논의에서 "5C에 안 맞는 게
    C군의 핵심 후보"라고 한 부분을 반영 (강제로 5C에 끼워맞추지 않음)

입력 : model_dataset_v2.csv, outputs_uni/univariate_summary.csv
출력 : outputs_uni/group_decision.csv       (개념그룹별 판정)
       변수선정_최종.csv                      (변수 단위 최종 포함/제외 + 근거)
"""

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content/drive/MyDrive/BOOSTMAP/데이터/원본")  # 본인 환경에 맞게 수정 (step1과 동일 경로)

DATA = BASE_DIR / "model_dataset_v2.csv"
SAMPLE_N = 150000          # 속도용 표본 (전체 30만이면 시간 오래 걸림)
RANDOM_STATE = 42

### 개념그룹 정의 (build_groups)

In [17]:
def build_groups(allcols):
    """변수를 '개념 그룹'으로 묶는다. 원핫은 접두사로 묶음."""
    g = {
        # ---- 기존 A_금융/B_비금융 그룹 (변경 없음) ----
        "ANNUITY_TO_CREDIT(상환부담)": ["ANNUITY_TO_CREDIT"],
        "DTI(부채상환비율)": ["DTI"],
        "CREDIT_TO_INCOME(대출/소득)": ["CREDIT_TO_INCOME"],
        "LTV_GOODS(담보비율)": ["LTV_GOODS"],
        "YEARS_EMPLOYED(근속)": ["YEARS_EMPLOYED"],
        "AGE(나이)": ["AGE"],
        "DAYS_EMPLOYED_ANOM(고용이상)": ["DAYS_EMPLOYED_ANOM"],
        "FLAG_OWN_REALTY(자가보유)": ["FLAG_OWN_REALTY_BIN"],
        "NAME_CONTRACT_TYPE(대출유형)": ["NAME_CONTRACT_TYPE_BIN"],
        "AMT_INCOME_OUTLIER(소득이상)": ["AMT_INCOME_OUTLIER"],
        "OCCUPATION(직업)": [c for c in allcols if c.startswith("OCCUPATION_TYPE")],
        "ORGANIZATION(업종)": [c for c in allcols if c.startswith("ORGANIZATION_TYPE")],
        "INCOME_TYPE(소득유형)": [c for c in allcols if c.startswith("NAME_INCOME_TYPE")],
        "AMT원본_금액군(중복후보)": ["AMT_CREDIT", "AMT_CREDIT_LOG", "AMT_ANNUITY",
                             "AMT_ANNUITY_LOG", "AMT_GOODS_PRICE", "AMT_GOODS_PRICE_LOG",
                             "AMT_INCOME_TOTAL", "AMT_INCOME_TOTAL_LOG"],

        # ---- (v2 신규) A_금융 추가 개념그룹 ----
        "AMT_REQ_CREDIT_BUREAU(신용조회횟수)": [c for c in allcols if c.startswith("AMT_REQ_CREDIT_BUREAU")],
        "BUREAU_이력(타기관대출이력)": ["BUREAU_LOAN_COUNT", "BUREAU_ACTIVE_LOAN_COUNT",
                                "BUREAU_DAYS_OVERDUE_MAX", "BUREAU_DAYS_OVERDUE_MEAN",
                                "BUREAU_DEBT_RATIO", "BUREAU_NO_HISTORY_FLAG"],

        # ---- (v2 신규) C_비금융_신규 개념그룹 ----
        "SOCIAL_CIRCLE(사회연결망·RQ2핵심)": ["OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE",
                                         "SOCIAL_CIRCLE_MISSING_FLAG"],
        "FLAG_CONT_MOBILE(연락가능성)": ["FLAG_CONT_MOBILE"],
        "CNT_CHILDREN(자녀수)": ["CNT_CHILDREN"],
        "FLAG_OWN_CAR(차량보유)": ["FLAG_OWN_CAR"],
        "NAME_EDUCATION_TYPE(교육수준)": [c for c in allcols if c.startswith("NAME_EDUCATION_TYPE_C_")],
        "NAME_HOUSING_TYPE(주거유형)": [c for c in allcols if c.startswith("NAME_HOUSING_TYPE_C_")],
    }
    used = set(sum(g.values(), []))
    leftover = [c for c in allcols if c not in used]
    if leftover:
        g["기타"] = leftover
    return g

### 5C 프레임워크 매핑 (map_5c)

In [18]:
def map_5c(v):
    # ---- (v2 신규) C군/bureau 명시적 매핑 — 순서상 최우선 처리 ----
    # 사회연결망·통신은 전통 5C 어디에도 안 맞는 게 핵심이라 강제로 끼워맞추지 않음
    if v.startswith(("OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE")) or v == "SOCIAL_CIRCLE_MISSING_FLAG":
        return "기타(5C밖 신규축·RQ2핵심)"
    if v == "FLAG_CONT_MOBILE":
        return "기타(5C밖 신규축)"
    if v == "CNT_CHILDREN":
        return "Capacity(상환능력)"
    if v == "FLAG_OWN_CAR":
        return "Collateral(담보)"
    if v.startswith("NAME_EDUCATION_TYPE_C_"):
        return "Capacity(상환능력)"
    if v.startswith("NAME_HOUSING_TYPE_C_"):
        return "Collateral(담보)"
    # bureau/신용조회횟수: 과거 상환이력·최근 신용활동 = Character(상환의지) 축 보강
    if v.startswith("AMT_REQ_CREDIT_BUREAU") or v.startswith("BUREAU_"):
        return "Character(상환의지)"

    # ---- 기존 로직 (변경 없음) ----
    if v.startswith(("OCCUPATION", "ORGANIZATION", "NAME_INCOME", "NAME_CONTRACT")):
        return "Conditions(여건)"
    if v == "FLAG_OWN_REALTY_BIN":
        return "Capital(자본)"
    if v == "AGE":
        return "Character(상환의지·대리)"
    if v in ["LTV_GOODS"] or "GOODS" in v or "CREDIT" in v:
        return "Collateral(담보)"
    return "Capacity(상환능력)"

### 실행 (main) — AUC 계산 → 그룹기여도 → 최종 판정

In [19]:
def main():
    df = pd.read_csv(DATA).sample(SAMPLE_N, random_state=1).reset_index(drop=True)
    y = df.TARGET.values
    s = pd.read_csv(BASE_DIR / "outputs_uni/univariate_summary.csv")
    iv = dict(zip(s.feature, s.IV)); ext = dict(zip(s.feature, s["EXT2_상관"]))
    grp = dict(zip(s.feature, s.group))

    excl = [c for c in df.columns if "EXT_SOURCE" in c] + ["SK_ID_CURR", "TARGET"]
    allcols = [c for c in df.columns if c not in excl]
    groups = build_groups(allcols)

    # 그룹이 실제 컬럼을 다 덮는지 확인 (v2 변경으로 인한 누락 방지)
    covered = set(sum(groups.values(), []))
    print(f"전체 피처 {len(allcols)}개 / 그룹으로 분류된 피처 {len(covered)}개")
    assert covered == set(allcols), f"미분류 컬럼 존재: {set(allcols) - covered}"

    folds = list(StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE).split(df, y))

    def auc(cols):
        if not cols:
            return 0.5
        X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        oof = np.zeros(len(y))
        for tr, va in folds:
            m = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.04, num_leaves=31,
                                   colsample_bytree=.8, subsample=.9, subsample_freq=1,
                                   verbose=-1, n_jobs=-1).fit(X.iloc[tr], y[tr])
            oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        return roc_auc_score(y, oof)

    base = auc(allcols)
    print(f"전체 AUC = {base:.5f}\n그룹별 leave-one-out 측정...")

    # 개념그룹별 기여도
    rows = []
    for gname, cols in groups.items():
        a_out = auc([c for c in allcols if c not in cols])
        rows.append({"개념그룹": gname, "변수수": len(cols),
                     "그룹기여도": round(base - a_out, 5)})
    gdf = pd.DataFrame(rows).sort_values("그룹기여도", ascending=False)
    gdf.to_csv(BASE_DIR / "outputs_uni/group_decision.csv", index=False, encoding="utf-8-sig")
    print(gdf.to_string(index=False))

    # 변수 단위 최종 판정표
    gc_map = dict(zip(gdf.개념그룹, gdf.그룹기여도))
    out = []
    for gname, cols in groups.items():
        gcontrib = gc_map[gname]
        is_onehot = len(cols) > 1 and "중복" not in gname
        for c in cols:
            if "중복후보" in gname:                       # 원본 금액: 대표 2개만
                keep = c in ["AMT_CREDIT_LOG", "AMT_INCOME_TOTAL_LOG"]
                judg = "대표유지(규모정보)" if keep else "제거(비율변수와 중복)"
                unit = "그룹(원본금액)"
            else:
                keep = gcontrib >= 0                       # 기여도 0 이상이면 포함
                judg = ("필수" if gcontrib >= 0.002 else
                        "포함권장" if gcontrib >= 0.0003 else
                        "선택(해석용)" if gcontrib >= 0 else "제거가능")
                unit = "그룹(원핫)" if is_onehot else "개별"
            out.append({"변수": c, "개념그룹": gname.split("(")[0],
                        "군": grp.get(c, "A_금융"), "5C_매핑": map_5c(c),
                        "IV": round(iv.get(c, np.nan), 4),
                        "EXT상관": round(ext.get(c, np.nan), 4),
                        "그룹기여도": round(gcontrib, 5), "평가단위": unit,
                        "최종판정": judg, "최종포함": "포함" if keep else "제외"})
    fdf = pd.DataFrame(out).sort_values(["최종포함", "그룹기여도"], ascending=[True, False])
    fdf.to_csv(BASE_DIR / "변수선정_최종.csv", index=False, encoding="utf-8-sig")
    print(f"\n최종: 포함 {(fdf.최종포함=='포함').sum()}개 / "
          f"제외 {(fdf.최종포함=='제외').sum()}개")


if __name__ == "__main__":
    main()

전체 피처 77개 / 그룹으로 분류된 피처 77개
전체 AUC = 0.70875
그룹별 leave-one-out 측정...
                         개념그룹  변수수    그룹기여도
           BUREAU_이력(타기관대출이력)    6  0.01418
      ANNUITY_TO_CREDIT(상환부담)    1  0.01301
           YEARS_EMPLOYED(근속)    1  0.00378
    NAME_EDUCATION_TYPE(교육수준)    5  0.00321
                      AGE(나이)    1  0.00153
               OCCUPATION(직업)   18  0.00083
     NAME_CONTRACT_TYPE(대출유형)    1  0.00070
   SOCIAL_CIRCLE(사회연결망·RQ2핵심)    3  0.00062
           FLAG_OWN_CAR(차량보유)    1  0.00059
AMT_REQ_CREDIT_BUREAU(신용조회횟수)    7  0.00051
            INCOME_TYPE(소득유형)    5  0.00036
              AMT원본_금액군(중복후보)    8  0.00032
              LTV_GOODS(담보비율)    1  0.00023
      NAME_HOUSING_TYPE(주거유형)    6  0.00009
             ORGANIZATION(업종)    6  0.00004
      CREDIT_TO_INCOME(대출/소득)    1  0.00003
     AMT_INCOME_OUTLIER(소득이상)    1  0.00000
        FLAG_OWN_REALTY(자가보유)    1 -0.00007
      FLAG_CONT_MOBILE(연락가능성)    1 -0.00021
                  DTI(부채상환비율)    1 -0.00024
     DA